In [7]:
# CELL 1 - Create Flask app
app_code = '''from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib
import numpy as np
from datetime import datetime

app = Flask(__name__)
CORS(app)

# Load models
model = joblib.load('best_model.pkl')
scaler = joblib.load('scaler.pkl')

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy', 'service': 'InternSpark ML API'})

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        features = np.array(data['features']).reshape(1, -1)
        features_scaled = scaler.transform(features)
        
        prediction = model.predict(features_scaled)[0]
        probability = model.predict_proba(features_scaled)[0]
        
        return jsonify({
            'prediction': int(prediction),
            'tumor_type': 'Benign' if prediction == 1 else 'Malignant',
            'confidence': float(probability[prediction]),
            'probabilities': {
                'malignant': float(probability[0]),
                'benign': float(probability[1])
            }
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
'''

with open('app.py', 'w') as f:
    f.write(app_code)
print("✅ Created app.py")

✅ Created app.py


In [8]:
# CELL 2 - Create requirements file
requirements = '''flask==2.3.3
flask-cors==4.0.0
numpy==1.24.3
scikit-learn==1.3.0
joblib==1.3.2
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements)
print("✅ Created requirements.txt")

✅ Created requirements.txt


In [9]:
# CELL 3 - Create Dockerfile
dockerfile = '''FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY best_model.pkl .
COPY scaler.pkl .

EXPOSE 5000

CMD ["python", "app.py"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)
print("✅ Created Dockerfile")

✅ Created Dockerfile


In [10]:
# CELL 4 - Instructions
print("""
🚀 HOW TO RUN YOUR API:

STEP 1: Open a NEW terminal (don't use Jupyter)
        (In Jupyter: Click 'New' -> 'Terminal')

STEP 2: Navigate to your project folder:
        cd C:\\Users\\aruhi

STEP 3: Install Flask (if not installed):
        pip install flask flask-cors

STEP 4: Run the API:
        python app.py

You should see:
    * Running on http://0.0.0.0:5000

STEP 5: Test the API (open ANOTHER terminal):
        
        curl -X POST http://localhost:5000/predict \\
          -H "Content-Type: application/json" \\
          -d '{"features": [17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.07871, 1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193, 25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189]}'

Expected Response:
    {
        "prediction": 0,
        "tumor_type": "Malignant",
        "confidence": 0.99,
        "probabilities": {
            "malignant": 0.99,
            "benign": 0.01
        }
    }
""")


🚀 HOW TO RUN YOUR API:

STEP 1: Open a NEW terminal (don't use Jupyter)
        (In Jupyter: Click 'New' -> 'Terminal')

STEP 2: Navigate to your project folder:
        cd C:\Users\aruhi

STEP 3: Install Flask (if not installed):
        pip install flask flask-cors

STEP 4: Run the API:
        python app.py

You should see:
    * Running on http://0.0.0.0:5000

STEP 5: Test the API (open ANOTHER terminal):

        curl -X POST http://localhost:5000/predict \
          -H "Content-Type: application/json" \
          -d '{"features": [17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.07871, 1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193, 25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189]}'

Expected Response:
    {
        "prediction": 0,
        "tumor_type": "Malignant",
        "confidence": 0.99,
        "probabilities": {
            "malignant": 0.99,
            "benign": 0.01
        }
    }

In [11]:
# CELL 5 - Test API using Python (if curl doesn't work)
import requests
import json

print("Testing API with Python...")

# Sample features (malignant case)
sample_features = [17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.07871,
                   1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
                   25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189]

try:
    response = requests.post(
        'http://localhost:5000/predict',
        json={'features': sample_features},
        timeout=5
    )
    
    if response.status_code == 200:
        print("\n✅ API Response:")
        print(json.dumps(response.json(), indent=2))
    else:
        print(f"Error: {response.status_code}")
        print(f"Make sure the API is running (python app.py)")
        
except requests.exceptions.ConnectionError:
    print("\n❌ Cannot connect to API!")
    print("Please start the API first by running in a separate terminal:")
    print("   python app.py")
except Exception as e:
    print(f"Error: {e}")

Testing API with Python...

❌ Cannot connect to API!
Please start the API first by running in a separate terminal:
   python app.py


In [12]:
# START API AND TEST - SIMPLE VERSION
import subprocess
import time
import requests
import json
import os

# Check if model exists
print("📁 Checking files...")
print(f"best_model.pkl: {'✅ FOUND' if os.path.exists('best_model.pkl') else '❌ MISSING'}")
print(f"scaler.pkl: {'✅ FOUND' if os.path.exists('scaler.pkl') else '❌ MISSING'}")
print(f"app.py: {'✅ FOUND' if os.path.exists('app.py') else '❌ MISSING'}")

if not all([os.path.exists('best_model.pkl'), os.path.exists('scaler.pkl'), os.path.exists('app.py')]):
    print("\n❌ Some files are missing. Run Task 1 and Task 4 first!")
else:
    print("\n✅ All files ready! Starting API...")
    
    # Start API
    process = subprocess.Popen(
        ['python', 'app.py'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        shell=True
    )
    
    # Wait for API to start
    print("⏳ Waiting for API to start...")
    time.sleep(5)
    
    # Test the API
    print("\n📡 Testing prediction...")
    
    sample_features = [
        17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.07871,
        1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
        25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189
    ]
    
    try:
        response = requests.post(
            'http://localhost:5000/predict',
            json={'features': sample_features},
            timeout=10
        )
        
        if response.status_code == 200:
            print("\n" + "="*50)
            print("🎉 SUCCESS! API IS WORKING!")
            print("="*50)
            result = response.json()
            print(f"\n📊 Prediction Result:")
            print(f"   Tumor Type: {result['tumor_type']}")
            print(f"   Confidence: {result['confidence']:.2%}")
            print(f"\n📋 Full Response:")
            print(json.dumps(result, indent=2))
        else:
            print(f"❌ Error: {response.status_code}")
            print(response.text)
            
    except requests.exceptions.ConnectionError:
        print("\n❌ Could not connect to API on port 5000")
        print("\nTry this alternative method:")
        print("1. Open Command Prompt (not Jupyter)")
        print("2. Type: python app.py")
        print("3. Then run the test cell again")
        
    except Exception as e:
        print(f"\n❌ Error: {e}")

📁 Checking files...
best_model.pkl: ❌ MISSING
scaler.pkl: ❌ MISSING
app.py: ✅ FOUND

❌ Some files are missing. Run Task 1 and Task 4 first!


In [13]:
import os

print("📍 CURRENT JUPYTER LOCATION:")
print(os.getcwd())
print("\n📁 FILES IN THIS FOLDER:")
for file in os.listdir('.'):
    if file.endswith('.pkl') or file.endswith('.ipynb') or file == 'app.py':
        print(f"   {file}")

📍 CURRENT JUPYTER LOCATION:
C:\Users\aruhi

📁 FILES IN THIS FOLDER:
    Task4_Deployment_API.ipynb
   app.py
   Task3_Responsible_AI.ipynb


In [14]:
# RUN THIS CELL - Shows prediction response for screenshot
import requests
import json
from datetime import datetime

print("="*60)
print("INTERNSPARK API - PREDICTION RESPONSE")
print("="*60)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Sample features (breast cancer case)
sample_features = [
    17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.07871,
    1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
    25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189
]

try:
    # Make API call
    response = requests.post(
        'http://localhost:5000/predict',
        json={'features': sample_features},
        timeout=5
    )
    
    if response.status_code == 200:
        result = response.json()
        
        print("📊 PREDICTION RESULT:")
        print("-" * 40)
        print(f"   Tumor Type: {result['tumor_type']}")
        print(f"   Confidence: {result['confidence']:.2%}")
        print()
        print("📋 FULL JSON RESPONSE:")
        print("-" * 40)
        print(json.dumps(result, indent=2))
        print("-" * 40)
        print("\n✅ API is working correctly!")
        
    else:
        print(f"❌ Error: {response.status_code}")
        
except Exception as e:
    print(f"❌ Cannot connect: {e}")
    print("Make sure API is running: python app.py")

print("="*60)


INTERNSPARK API - PREDICTION RESPONSE
Timestamp: 2026-06-08 19:52:03

📊 PREDICTION RESULT:
----------------------------------------
   Tumor Type: Malignant
   Confidence: 96.00%

📋 FULL JSON RESPONSE:
----------------------------------------
{
  "confidence": 0.96,
  "prediction": 0,
  "probabilities": {
    "benign": 0.04,
    "malignant": 0.96
  },
  "tumor_type": "Malignant"
}
----------------------------------------

✅ API is working correctly!
